<a href="https://colab.research.google.com/github/galvezam/eia-data-pipeline/blob/main/ingest/total_ingest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install boto3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 2.5 MB/s eta 0:00:00


In [ ]:
import io
import csv
import json
import time
import argparse
import requests
import boto3
from concurrent.futures import ThreadPoolExecutor, as_completed
from dateutil.relativedelta import relativedelta
from datetime import date, datetime

from botocore.exceptions import NoCredentialsError
from google.colab import userdata

AWS_ACCESS_KEY = userdata.get("AWS_ACCESS_KEY")
AWS_SECRET_KEY = userdata.get("AWS_SECRET_KEY")
AWS_REGION     = userdata.get("AWS_REGION")
BUCKET_NAME    = userdata.get("BUCKET_NAME")
EIA_API_KEY    = userdata.get("EIA_API_KEY")

s3_client = boto3.client(
    "s3",
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    region_name=AWS_REGION,
)

In [ ]:
DATASETS = {

    # Core energy datasets

    "petroleum": {
        "label": "Petroleum Production",
        "url": "https://api.eia.gov/v2/petroleum/pnp/wprodrb/data/",
        "frequency": "weekly",
        "data": ["value"],
        "facets": {},
        "start": "2025-01-01",
        "end": "2025-12-31",
        "page_size": 5000,
        "max_page_workers": 3,
        "max_date_workers": 3,
        "date_partition": True,
        "incremental_frequency": "1 month",
    },
    "natural_gas": {
        "label": "Natural Gas Monthly Production",
        "url": "https://api.eia.gov/v2/natural-gas/prod/whv/data/",
        "frequency": "monthly",
        "data": ["value"],
        "facets": {},
        "start": "2025-01",
        "end": "2025-12",
        "page_size": 5000,
        "max_page_workers": 3,
        "max_date_workers": 3,
        "date_partition": True,
        "incremental_frequency": "2 months",
    },
    "electricity": {
        "label": "Electricity Operations",
        "url": "https://api.eia.gov/v2/electricity/electric-power-operational-data/data/",
        "frequency": "monthly",
        "data": ["consumption-for-eg", "consumption-for-eg-btu"],
        "facets": {
            "fueltypeid": ["COL", "NG", "NUC", "HYC", "WND", "SUN", "GEO", "ALL", "AOR"],
        },
        "start": "2025-01",
        "end": "2025-12",
        "page_size": 5000,
        "max_page_workers": 3,
        "max_date_workers": 3,
        "date_partition": True,
        "incremental_frequency": "2 months",
    },
    "crude_oil_imports": {
        "label": "Crude Oil Imports",
        "url": "https://api.eia.gov/v2/crude-oil-imports/data/",
        "frequency": "monthly",
        "data": ["quantity"],
        "facets": {},
        "start": "2025-01",
        "end": "2025-12",
        "page_size": 5000,
        "max_page_workers": 3,
        "max_date_workers": 3,
        "date_partition": True,
        "incremental_frequency": "2 months",
    },

    # Macro / outlook

    "steo": {
        "label": "Short Term Energy Outlook",
        "url": "https://api.eia.gov/v2/steo/data/",
        "frequency": "monthly",
        "data": ["value"],
        "facets": {},
        "start": "2025-01",
        "end": "2025-12",
        "page_size": 5000,
        "max_page_workers": 3,
        "max_date_workers": 3,
        "date_partition": True,
        "incremental_frequency": "2 months",
    },
    "total_energy": {
        "label": "Total Energy",
        "url": "https://api.eia.gov/v2/total-energy/data/",
        "frequency": "monthly",
        "data": ["value"],
        "facets": {},
        "start": "2025-01",
        "end": "2025-12",
        "page_size": 5000,
        "max_page_workers": 3,
        "max_date_workers": 3,
        "date_partition": True,
        "incremental_frequency": "2 months",
    },
    "seds": {
        "label": "State Energy Data System",
        "url": "https://api.eia.gov/v2/seds/data/",
        "frequency": "annual",
        "data": ["value"],
        "facets": {},
        "start": "2024",
        "end": "2025",
        "page_size": 5000,
        "max_page_workers": 3,
        "max_date_workers": 3,
        "date_partition": True,
        "incremental_frequency": "1 year",
    },

    # Coal

    "coal": {
        "label": "Coal Import / Export",
        "url": "https://api.eia.gov/v2/coal/exports-imports-quantity-price/data/",
        "frequency": "annual",
        "data": ["price", "quantity"],
        "facets": {},
        "start": "2024",
        "end": "2025",
        "page_size": 5000,
        "max_page_workers": 3,
        "max_date_workers": 3,
        "date_partition": True,
        "incremental_frequency": "1 year",
    },

    # Natural gas trade

    "natural_gas_trade": {
        "label": "Natural Gas Import / Export by State",
        "url": "https://api.eia.gov/v2/natural-gas/move/ist/data/",
        "frequency": "annual",
        "data": ["value"],
        "facets": {},
        "start": "2024",
        "end": "2025",
        "page_size": 5000,
        "max_page_workers": 3,
        "max_date_workers": 3,
        "date_partition": True,
        "incremental_frequency": "1 year",
    },

    # State electricity profiles

    "electricity_state_rankings": {
        "label": "State Electricity Rankings",
        "url": "https://api.eia.gov/v2/electricity/state-electricity-profiles/summary/data/",
        "frequency": "annual",
        "data": [
            "average-retail-price-rank",
            "carbon-dioxide-rank",
            "direct-use-rank",
            "fsp-sales-rank",
            "generation-elect-utils-rank",
            "net-generation-rank",
            "nitrogen-oxide-rank",
            "sulfer-dioxide-rank",
        ],
        "facets": {},
        "start": None,
        "end": None,
        "page_size": 5000,
        "max_page_workers": 3,
        "max_date_workers": 3,
        "date_partition": False,
        "incremental_frequency": "1 year",
    },
    "electricity_net_metering": {
        "label": "State Electricity Net Metering",
        "url": "https://api.eia.gov/v2/electricity/state-electricity-profiles/net-metering/data/",
        "frequency": "annual",
        "data": ["capacity", "customers"],
        "facets": {},
        "start": "2024",
        "end": "2025",
        "page_size": 5000,
        "max_page_workers": 3,
        "max_date_workers": 3,
        "date_partition": True,
        "incremental_frequency": "1 year",
    },
    "electricity_generating_capacity": {
        "label": "State Electricity Generating Capacity",
        "url": "https://api.eia.gov/v2/electricity/state-electricity-profiles/capability/data/",
        "frequency": "annual",
        "data": ["capability"],
        "facets": {},
        "start": "2024",
        "end": "2025",
        "page_size": 5000,
        "max_page_workers": 3,
        "max_date_workers": 3,
        "date_partition": True,
        "incremental_frequency": "1 year",
    },

    # Biomass

    "densified_biomass": {
        "label": "Densified Biomass Export Prices & Quantity",
        "url": "https://api.eia.gov/v2/densified-biomass/export-sales-and-price/data/",
        "frequency": "monthly",
        "data": ["average-price", "quantity"],
        "facets": {},
        "start": "2025-01",
        "end": "2025-12",
        "page_size": 5000,
        "max_page_workers": 3,
        "max_date_workers": 3,
        "date_partition": True,
        "incremental_frequency": "2 months",
    },

    # Petroleum movements

    "petroleum_movements": {
        "label": "Petroleum Imports / Exports & Movements",
        "url": "https://api.eia.gov/v2/petroleum/move/wkly/data/",
        "frequency": "weekly",
        "data": ["value"],
        "facets": {},
        "start": "2025-01-01",
        "end": "2026-12-31",
        "page_size": 5000,
        "max_page_workers": 3,
        "max_date_workers": 3,
        "date_partition": True,
        "incremental_frequency": "1 month",
    },
}

In [ ]:
from pathlib import Path

# Either s3, local, or both
WRITE_MODE = "local"
BASE_OUTPUT = "data"

def get_local_path(dataset_name: str, config: dict) -> str:
    today = datetime.utcnow().strftime("%Y%m%d")
    path = Path(BASE_OUTPUT) / dataset_name / config["frequency"]
    path.mkdir(parents=True, exist_ok=True)
    return str(path / f"{dataset_name}_{today}.csv")

def stream_data(dataset_name: str, config: dict):
    write_s3 = WRITE_MODE in ("s3", "both")
    write_local = WRITE_MODE in ("local", "both")

    s3_key = f"raw/{dataset_name}/{config['frequency']}/{dataset_name}.csv"
    local_path = get_local_path(dataset_name, config)

    # S3 setup
    if write_s3:
        multipart = s3_client.create_multipart_upload(
            Bucket=BUCKET_NAME, Key=s3_key
        )
        parts = []
        part_number = 1

    # Local setup
    if write_local:
        local_file = open(local_path, "w", newline="")
        local_writer = None

    buffer = io.StringIO()
    header_written = False
    fieldnames = None

    FLUSH_SIZE = 20 * 1024 * 1024

    def flush_buffer():
        nonlocal buffer, part_number

        data = buffer.getvalue()

        if write_s3:
            part = s3_client.upload_part(
                Bucket=BUCKET_NAME,
                Key=s3_key,
                PartNumber=part_number,
                UploadId=multipart["UploadId"],
                Body=data.encode("utf-8"),
            )
            parts.append({"PartNumber": part_number, "ETag": part["ETag"]})
            part_number += 1

        if write_local:
            local_file.write(data)

        buffer = io.StringIO()

    def write_records(records: list):
        nonlocal header_written, fieldnames, local_writer

        if not header_written:
            fieldnames = list(records[0].keys())

            writer = csv.DictWriter(buffer, fieldnames=fieldnames)
            writer.writeheader()

            if write_local:
                local_writer = csv.DictWriter(local_file, fieldnames=fieldnames)
                local_writer.writeheader()

            header_written = True
        else:
            writer = csv.DictWriter(buffer, fieldnames=fieldnames, extrasaction="ignore")

        writer.writerows(records)

        if write_local:
            local_writer.writerows(records)

        if buffer.tell() > FLUSH_SIZE:
            flush_buffer()

In [ ]:
# Incremental helpers

def get_latest_period(dataset_name: str, config: dict) -> str | None:
    """
    Get latest period from either S3 or local depending on WRITE_MODE.
    """
    if WRITE_MODE in ("s3", "both"):
        try:
            s3_key = f"raw/{dataset_name}/{config['frequency']}/{dataset_name}.csv"
            obj = s3_client.get_object(Bucket=BUCKET_NAME, Key=s3_key)
            body = obj["Body"].read().decode("utf-8")
            reader = csv.DictReader(io.StringIO(body))
            periods = [row["period"] for row in reader if row.get("period")]
            return sorted(periods)[-1] if periods else None
        except Exception:
            pass

    if WRITE_MODE in ("local", "both"):
        path = Path(BASE_OUTPUT) / "raw" / dataset_name / config["frequency"]
        if not path.exists():
            return None

        files = sorted(path.glob("*.csv"))
        if not files:
            return None

        latest_file = files[-1]
        with open(latest_file, "r") as f:
            reader = csv.DictReader(f)
            periods = [row["period"] for row in reader if row.get("period")]
            return sorted(periods)[-1] if periods else None

    return None


def compute_incremental_start(latest_period: str, config: dict) -> str:
    """
    Given the latest period already stored, return the start date to use
    for the next incremental fetch (one period before latest to catch
    any late-arriving revisions).
    """
    freq = config["incremental_frequency"]
    amount, unit = freq.split()
    amount = int(amount)

    if config["frequency"] == "annual":
        yr = int(latest_period[:4])
        return str(yr - amount)

    # Parse as a date — handles YYYY-MM and YYYY-MM-DD
    period_str = latest_period
    if len(period_str) == 7:
        period_str += "-01"
    dt = date.fromisoformat(period_str)

    if unit in ("month", "months"):
        dt -= relativedelta(months=amount)
        return dt.strftime("%Y-%m")
    elif unit in ("week", "weeks"):
        dt -= relativedelta(weeks=amount)
        return dt.strftime("%Y-%m-%d")
    elif unit in ("year", "years"):
        dt -= relativedelta(years=amount)
        return str(dt.year) if config["frequency"] == "annual" else dt.strftime("%Y-%m")

    return latest_period


# Partition helpers

def generate_partitions(start: str, end: str, frequency: str) -> list[tuple[str, str]]:
    """
    Split a date range into per-period (start, end) tuples.
      annual         → one partition per year   ('2024', '2024')
      monthly/weekly → one partition per month  ('2024-01', '2024-01')
    """
    if frequency == "annual":
        start_yr = int(start[:4])
        end_yr   = int(end[:4])
        return [(str(y), str(y)) for y in range(start_yr, end_yr + 1)]

    def to_month(s):
        parts = s.split("-")
        return f"{parts[0]}-{parts[1]}"

    partitions = []
    current = date.fromisoformat(to_month(start) + "-01")
    end_dt  = date.fromisoformat(to_month(end)   + "-01")
    while current <= end_dt:
        m = current.strftime("%Y-%m")
        partitions.append((m, m))
        current += relativedelta(months=1)
    return partitions


# API request helpers

def build_params(config: dict, offset: int, start: str | None, end: str | None) -> dict:
    p = {
        "api_key": EIA_API_KEY,
        "frequency": config["frequency"],
        "offset": offset,
        "length": config["page_size"],
        "sort[0][column]": "period",
        "sort[0][direction]": "desc",
    }
    if start is not None:
        p["start"] = start
    if end is not None:
        p["end"] = end
    for i, d in enumerate(config["data"]):
        p[f"data[{i}]"] = d
    return p


def build_headers(config: dict, offset: int, start: str | None, end: str | None) -> dict:
    return {
        "X-Params": json.dumps({
            "frequency": config["frequency"],
            "data": config["data"],
            "facets": config.get("facets", {}),
            "start": start,
            "end": end,
            "sort": [{"column": "period", "direction": "desc"}],
            "offset": offset,
            "length": config["page_size"],
        })
    }


def fetch_page(config: dict, offset: int, start: str | None, end: str | None) -> dict:
    max_retries = 5

    for attempt in range(max_retries):
        response = requests.get(
            config["url"],
            params=build_params(config, offset, start, end),
            headers=build_headers(config, offset, start, end),
            timeout=60,
        )
        if response.status_code == 200:
            return response.json().get("response", {})

        body = response.text
        is_rate_limit = response.status_code == 429 or "OVER_RATE_LIMIT" in body

        if is_rate_limit and attempt < max_retries - 1:
            wait = 2 * (2 ** attempt)
            print(f"  Rate limited [{start}→{end}] offset={offset} "
                  f"— retrying in {wait}s (attempt {attempt + 1}/{max_retries})")
            time.sleep(wait)
            continue

        raise Exception(f"offset={offset} [{start}→{end}]: {body}")

    raise Exception(f"offset={offset} [{start}→{end}]: max retries exceeded")


def fetch_all_pages(dataset_name: str, config: dict, start: str | None, end: str | None) -> list:
    """Fetch page 0 for total count, then fan out remaining pages in parallel."""
    first = fetch_page(config, 0, start, end)
    total = int(first.get("total", 0))
    first_records = first.get("data", [])

    if not first_records:
        return []

    all_records = list(first_records)
    remaining_offsets = list(range(config["page_size"], total, config["page_size"]))

    print(f"  {dataset_name} [{start}→{end}]: "
          f"page 0 → {len(first_records):,} records (total: {total:,})")

    if remaining_offsets:
        page_results = {}
        with ThreadPoolExecutor(max_workers=config["max_page_workers"]) as ex:
            futures = {
                ex.submit(fetch_page, config, offset, start, end): offset
                for offset in remaining_offsets
            }
            for future in as_completed(futures):
                offset = futures[future]
                records = future.result().get("data", [])
                page_results[offset] = records
                print(f"  {dataset_name} [{start}→{end}]: "
                      f"page {offset} → {len(records):,} records")

        for offset in remaining_offsets:
            all_records.extend(page_results.get(offset, []))

    return all_records


# Core streaming function

def stream_data(dataset_name: str, config: dict, incremental: bool = False):
    write_s3   = WRITE_MODE in ("s3", "both")
    write_local = WRITE_MODE in ("local", "both")

    # Paths
    s3_key = f"raw/{dataset_name}/{config['frequency']}/{dataset_name}.csv"

    today = datetime.utcnow().strftime("%Y%m%d")
    local_dir = Path(BASE_OUTPUT) / "raw" / dataset_name / config["frequency"]
    local_dir.mkdir(parents=True, exist_ok=True)
    local_path = local_dir / f"{dataset_name}_{today}.csv"

    # Incremental logic
    run_start = config["start"]
    run_end   = config["end"]

    if incremental:
        latest = get_latest_period(dataset_name, config)
        if latest:
            run_start = compute_incremental_start(latest, config)
            print(f"  [{dataset_name}] incremental from {run_start}")
        else:
            print(f"  [{dataset_name}] no existing data → full ingest")

    if incremental and run_start != config["start"]:
        s3_key = f"raw/{dataset_name}/{config['frequency']}/incremental/{dataset_name}_{today}.csv"

    # S3 init
    if write_s3:
        multipart = s3_client.create_multipart_upload(Bucket=BUCKET_NAME, Key=s3_key)
        parts = []
        part_number = 1

    # Local init
    if write_local:
        local_file = open(local_path, "w", newline="")
        local_writer = None

    buffer = io.StringIO()
    header_written = False
    fieldnames = None
    FLUSH_SIZE = 20 * 1024 * 1024

    def flush_buffer():
        nonlocal buffer, part_number
        data = buffer.getvalue()

        if write_s3:
            part = s3_client.upload_part(
                Bucket=BUCKET_NAME,
                Key=s3_key,
                PartNumber=part_number,
                UploadId=multipart["UploadId"],
                Body=data.encode("utf-8"),
            )
            parts.append({"PartNumber": part_number, "ETag": part["ETag"]})
            part_number += 1

        if write_local:
            local_file.write(data)

        buffer = io.StringIO()

    def write_records(records: list):
        nonlocal header_written, fieldnames, local_writer

        if not header_written:
            fieldnames = list(records[0].keys())

            writer = csv.DictWriter(buffer, fieldnames=fieldnames)
            writer.writeheader()

            if write_local:
                local_writer = csv.DictWriter(local_file, fieldnames=fieldnames)
                local_writer.writeheader()

            header_written = True
        else:
            writer = csv.DictWriter(buffer, fieldnames=fieldnames, extrasaction="ignore")

        writer.writerows(records)

        if write_local:
            local_writer.writerows(records)

        if buffer.tell() > FLUSH_SIZE:
            flush_buffer()

    # Fetch and write
    try:
        print(f"\n{'='*60}")
        print(f"{dataset_name} ({config['frequency']}) [{'INCR' if incremental else 'FULL'}]")
        print(f"{'='*60}")

        if config.get("date_partition") and run_start is not None:
            partitions = generate_partitions(run_start, run_end, config["frequency"])

            partition_results = {}
            with ThreadPoolExecutor(max_workers=config["max_date_workers"]) as ex:
                futures = {
                    ex.submit(fetch_all_pages, dataset_name, config, s, e): (s, e)
                    for s, e in partitions
                }
                for future in as_completed(futures):
                    partition_results[futures[future]] = future.result()

            for s, e in partitions:
                records = partition_results.get((s, e), [])
                if records:
                    write_records(records)

        else:
            records = fetch_all_pages(dataset_name, config, run_start, run_end)
            if records:
                write_records(records)

        if buffer.tell() > 0:
            flush_buffer()

        # finalize
        if write_local:
            local_file.close()
            print(f"✓ local → {local_path}")

        if write_s3 and parts:
            s3_client.complete_multipart_upload(
                Bucket=BUCKET_NAME,
                Key=s3_key,
                UploadId=multipart["UploadId"],
                MultipartUpload={"Parts": parts},
            )
            print(f"✓ s3 → s3://{BUCKET_NAME}/{s3_key}")

    except Exception as e:
        if write_s3:
            s3_client.abort_multipart_upload(
                Bucket=BUCKET_NAME,
                Key=s3_key,
                UploadId=multipart["UploadId"],
            )
        raise

In [ ]:
# Runner

def run_all(datasets: dict, incremental: bool = False):
    """
    Run ingestion for the given datasets dict.
    Cap top-level concurrency to avoid EIA rate limits.
    """
    MAX_CONCURRENT_DATASETS = 3
    print(f"\nRunning {'incremental' if incremental else 'full'} ingest "
          f"for {len(datasets)} dataset(s): {', '.join(datasets)}")

    with ThreadPoolExecutor(max_workers=MAX_CONCURRENT_DATASETS) as executor:
        futures = {
            executor.submit(stream_data, name, cfg, incremental): name
            for name, cfg in datasets.items()
        }
        for future in as_completed(futures):
            name = futures[future]
            try:
                future.result()
            except Exception as e:
                print(f"✗ {name} failed: {e}")



In [ ]:
# CLI entry point

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="EIA data ingestion pipeline")
    parser.add_argument(
        "--datasets",
        nargs="+",
        metavar="DATASET",
        help=(
            "Only ingest these datasets (space-separated). "
            f"Available: {', '.join(DATASETS)}"
        ),
    )
    parser.add_argument(
        "--incremental",
        action="store_true",
        help=(
            "Incremental mode: check what's already in S3 and only fetch "
            "newer periods. Writes to a dated incremental/ subfolder."
        ),
    )
    args, _ = parser.parse_known_args()

    # Filter datasets if --datasets was supplied
    if args.datasets:
        unknown = set(args.datasets) - set(DATASETS)
        if unknown:
            print(f"Unknown dataset(s): {', '.join(unknown)}")
            print(f"Available: {', '.join(DATASETS)}")
            raise SystemExit(1)
        selected = {k: DATASETS[k] for k in args.datasets}
    else:
        selected = DATASETS

    run_all(selected, incremental=args.incremental)



Running full ingest for 14 dataset(s): petroleum, natural_gas, electricity, crude_oil_imports, steo, total_energy, seds, coal, natural_gas_trade, electricity_state_rankings, electricity_net_metering, electricity_generating_capacity, densified_biomass, petroleum_movements

petroleum (weekly) [FULL]

electricity (monthly) [FULL]

natural_gas (monthly) [FULL]


/tmp/ipykernel_20995/1080350285.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = datetime.utcnow().strftime("%Y%m%d")


  natural_gas [2025-01→2025-01]: page 0 → 37 records (total: 37)
  natural_gas [2025-03→2025-03]: page 0 → 37 records (total: 37)
  natural_gas [2025-02→2025-02]: page 0 → 37 records (total: 37)
  natural_gas [2025-04→2025-04]: page 0 → 37 records (total: 37)
  natural_gas [2025-06→2025-06]: page 0 → 37 records (total: 37)
  natural_gas [2025-05→2025-05]: page 0 → 37 records (total: 37)
  natural_gas [2025-07→2025-07]: page 0 → 37 records (total: 37)
  natural_gas [2025-08→2025-08]: page 0 → 36 records (total: 36)
  electricity [2025-02→2025-02]: page 0 → 4,236 records (total: 4,236)
  natural_gas [2025-09→2025-09]: page 0 → 37 records (total: 37)
  petroleum [2025-08→2025-08]: page 0 → 109 records (total: 109)
  electricity [2025-01→2025-01]: page 0 → 4,238 records (total: 4,238)
  natural_gas [2025-10→2025-10]: page 0 → 37 records (total: 37)
  natural_gas [2025-11→2025-11]: page 0 → 37 records (total: 37)
  electricity [2025-03→2025-03]: page 0 → 4,234 records (total: 4,234)
  natur